In [1]:
#NOTE: MAKE SURE GUROBI IS INSTALLED AND YOU HAVE ACCESS TO GUROBIPY
from helper_functions import LSE, construct_R, decomp_orthog
import numpy as np
import gurobipy as gp
from gurobipy import GRB

In [2]:
#This function, given an NxI np-array Mat, creates a new NXM np array Mat_twoway which contains all main-effects, quadratic terms, and
#two-way interactions of the columns in Mat
def two_way_interaction(Mat):

    #Retrieve the number of columns in Mat
    num_cols_Mat = np.shape(Mat)[1]
    
    Mat_twoway = np.concatenate((Mat,Mat**2),axis = 1)
    for i in range(num_cols_Mat):
        for j in range(i+1, num_cols_Mat):
            interact_ij = np.array([Mat[:,i]*Mat[:,j]]).T
            Mat_twoway = np.concatenate((Mat_twoway,interact_ij), axis = 1)

    return Mat_twoway

In [3]:
#The three most important functions in this project are LSE, construct_R, and decomp_orthog.
#Note that decomp_orthog relies on the output of construct_R, and construct_R relies on LSE.

#We will start with showing how LSE works. LSE takes as input a model matrix X (an NxM numpy array, where N is the number of observations
#and M is the number of model terms) and a vector of responses Y. The output is a vector of coefficients corresponding to each model term.

#Some common models types could be: main effects model, quadratic effects model, and response surface model 
#(quadratic effects model with interaction terms)

#Below, we have some settings where you can choose the number of inputs and numbers of observations, respectively.
rng = np.random.default_rng(100)
num_inputs = 6 #IF THIS IS CHANGED, YOU MUST MAKE CORRESPONDING CHANGES TO vector_1 AND vector_2 IN THE decomp_orthog EXAMPLE.
num_obs = 250

#We can create a model matrix using only main effects. The inputs are assumed to come from unif[-1,1].
main_effects = rng.uniform(-1.0,1.0,size = (num_obs,num_inputs))

#We can also create a model matrix which has quadratic effects, given a model matrix of main effects
quadratic_effects = np.concatenate((main_effects,main_effects**2),axis = 1)

#Using the function "two_way_interaction", we can also create a model matrix having main effects, interactions, and quadratic terms.
interaction_effects = two_way_interaction(main_effects)

#Here, we will create a vector of responses
responses = rng.normal(0,1,size = num_obs)

#Next, we can fit different models to the responses using LSE.

main_effect_model = LSE(main_effects,responses)
print("main effect model coefficients:" +str(main_effect_model))

quad_effect_model = LSE(quadratic_effects,responses)
print("quad effect model coefficients:" + str(quad_effect_model))

int_effect_model = LSE(interaction_effects,responses)
print("int effect model coefficients:" + str(int_effect_model))

main effect model coefficients:[ 0.05265983 -0.01455678  0.0658577   0.18568133  0.04036267 -0.02008055]
quad effect model coefficients:[ 0.06506189 -0.03022629  0.06915419  0.1838642   0.06025212  0.01116743
 -0.11222238 -0.20514463 -0.24054868  0.08130988 -0.11862577  0.42558375]
int effect model coefficients:[ 0.05238221  0.03409394  0.02089961  0.23003115  0.08548677 -0.02336969
 -0.28521432 -0.12781216 -0.26632558  0.20820049 -0.07253719  0.47959626
 -0.19108034 -0.04304981 -0.14323306  0.03422548 -0.36048983 -0.10809763
 -0.05562436 -0.35879735 -0.36470501 -0.43045096 -0.16918003  0.07467036
 -0.15689825  0.11662407 -0.27179917]


In [4]:
#Having demonstrated the use of LSE, we will move on to construct_R. construct_R is crucial for running the function decomp_orthog.
#construct_R creates an NxM numpy matrix R where each entry is X_{n,m}*LSE_{m}. Here X is a NxM model matrix and LSE is an array of coefficient 
#estimates for each model term. Similar to LSE, construct_R takes in as input a model matrix X and response vector Y. construct_R returns both
#the LSE-scaled model matrix R and the LSE estimates as well.

R_main_effect = construct_R(main_effects,responses)
print("LSE-scaled model matrix R: " + str(R_main_effect[0]))
print("LSE:" + str(R_main_effect[1]))

R_quad_effect = construct_R(quadratic_effects,responses)
print("LSE-scaled model matrix R: " + str(R_quad_effect[0]))
print("LSE:" + str(R_quad_effect[1]))

R_int_effect = construct_R(interaction_effects,responses)
print("LSE-scaled model matrix R: " + str(R_int_effect[0]))
print("LSE:" + str(R_int_effect[1]))


LSE-scaled model matrix R: [[ 3.52811541e-02 -2.81003082e-03 -2.78089641e-02 -1.69729717e-01
   3.82369084e-02 -3.87340898e-03]
 [ 3.05714201e-02 -1.19454375e-02  2.47838397e-02 -1.15124588e-01
   3.88685507e-02  8.64607466e-03]
 [ 1.36160073e-02 -2.35825985e-03  1.31609858e-02  1.30908299e-02
   4.00227714e-02 -7.71568866e-05]
 ...
 [-1.02749578e-02 -1.00933090e-02 -1.05411659e-02  1.69932635e-02
  -3.68178423e-02 -1.57592219e-02]
 [-5.01095687e-03 -7.66257901e-03  6.37100167e-02  3.21643214e-03
  -6.19939977e-03 -1.24942222e-02]
 [ 3.57756117e-02  8.00754662e-03  1.50445840e-02 -7.83316790e-02
  -2.54598718e-02 -1.52658274e-02]]
LSE:[ 0.05265983 -0.01455678  0.0658577   0.18568133  0.04036267 -0.02008055]
LSE-scaled model matrix R: [[ 4.35900762e-02 -5.83594031e-03 -2.92009840e-02 ...  6.79413428e-02
  -1.06453043e-01  1.58442737e-02]
 [ 3.77711403e-02 -2.48050754e-02  2.60243378e-02 ...  3.12582359e-02
  -1.09999253e-01  7.88818341e-02]
 [ 1.68225136e-02 -4.89786445e-03  1.38197051e

In [7]:
#Set number of objective functions.
num_obj_fun = 4

#Before moving on to showing how one would setup and use decomp_orthog, we must first discuss strong heredity and how it is
#implemented in decomp_orthog.

#Implementation of strong heredity constraints in decomp_orthog requires the user to know the strong heredity relations between their model terms.
#decomp_orthog takes as an argument D, which is a list of lists of lists. Each list D_j in D corresponds to a response/objective function, and each
#list D_jm in D_j corresponds to indices of the parent effects of the mth model term.

#For a main effects model, there are no strong heredity relations. Thus, for the case where there are *num_inputs* inputs
#and *num_obj_fun* objective functions, the strong heredity list looks like
main_effect_strong_heredity = [[[] for i in range(num_inputs)] for j in range(num_obj_fun)]
print("strong heredity for main effects model: " +str(main_effect_strong_heredity))
#Note that the lists are all empty, since there are no strong heredity relations in a main effects model!

#For a quadratic effects model, there are strong heredity relations. Namely, if quadratic terms are included, then their main-effect counterparts 
#should be included as well.
quad_heredity = []
for i in range(num_inputs):
    quad_heredity.append([])
for i in range(num_inputs):
    quad_heredity.append([i])
quad_effect_strong_heredity = [quad_heredity for j in range(num_obj_fun)]
print("strong heredity for quadratic effects model: " +str(quad_effect_strong_heredity))

#Lastly, for a quadratic and interaction effects model, there are also strong heredity relations. If an interaction effect is included
#Then both of its parents effects should be included.An example of this is given below
int_heredity = []
for i in range(num_inputs):
    int_heredity.append([])
for i in range(num_inputs):
    int_heredity.append([i])
for i in range(num_inputs):
    for i_2 in range(i+1, num_inputs):
        int_heredity.append([i,i_2])
int_effect_strong_heredity = [int_heredity for j in range(num_obj_fun)]
print("strong heredity for interaction+quadratic effects model: " +str(int_effect_strong_heredity))

strong heredity for main effects model: [[[], [], [], [], [], []], [[], [], [], [], [], []], [[], [], [], [], [], []], [[], [], [], [], [], []]]
strong heredity for quadratic effects model: [[[], [], [], [], [], [], [0], [1], [2], [3], [4], [5]], [[], [], [], [], [], [], [0], [1], [2], [3], [4], [5]], [[], [], [], [], [], [], [0], [1], [2], [3], [4], [5]], [[], [], [], [], [], [], [0], [1], [2], [3], [4], [5]]]
strong heredity for interaction+quadratic effects model: [[[], [], [], [], [], [], [0], [1], [2], [3], [4], [5], [0, 1], [0, 2], [0, 3], [0, 4], [0, 5], [1, 2], [1, 3], [1, 4], [1, 5], [2, 3], [2, 4], [2, 5], [3, 4], [3, 5], [4, 5]], [[], [], [], [], [], [], [0], [1], [2], [3], [4], [5], [0, 1], [0, 2], [0, 3], [0, 4], [0, 5], [1, 2], [1, 3], [1, 4], [1, 5], [2, 3], [2, 4], [2, 5], [3, 4], [3, 5], [4, 5]], [[], [], [], [], [], [], [0], [1], [2], [3], [4], [5], [0, 1], [0, 2], [0, 3], [0, 4], [0, 5], [1, 2], [1, 3], [1, 4], [1, 5], [2, 3], [2, 4], [2, 5], [3, 4], [3, 5], [4, 5]],

In [13]:
#The last function we will look at is decomp_orthog. GUROBI and GUROBIPY are needed to run this function. 

#First, we will need a set of responses. This will take the form of a JxN matrix (J is the number of objective functions, and N is the number
#of observations). 

#vector_1 and vector_2 denote active and inactive effects for two clusters, here we are assuming a interaction effect model. Coefficient of active effects
#is 1.

#NOTE THAT FOR APPLICATION TO REAL DATA, ONE MAY NEED TO STANDARDIZE THE INPUTS AND OUTPUTS, AS WELL AS SUBTRACT OFF AN ESTIMATE OF THE INTERCEPT TERM
#FROM THE RESPONSES, AS decomp_orthog DOES NOT TAKE INTO ACCOUNT THE INTERCEPT TERM.
vector_1 = np.array([1,1,1,0,0,0,1,1,1,0,0,0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0])
vector_2 = np.array([0,0,0,1,1,1,0,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1])
epsilon_1 = rng.normal(0,1,size = num_obs)
epsilon_2 = rng.normal(0,1,size = num_obs)
epsilon_3 = rng.normal(0,1,size = num_obs)
epsilon_4 = rng.normal(0,1,size = num_obs)

response_1 = [np.dot(vector_1,interaction_effects[i,:]) + epsilon_1[i] for i in range(num_obs)]
response_2 = [np.dot(vector_1,interaction_effects[i,:]) + epsilon_2[i] for i in range(num_obs)]
response_3 = [np.dot(vector_2,interaction_effects[i,:]) + epsilon_3[i] for i in range(num_obs)]
response_4 = [np.dot(vector_2,interaction_effects[i,:]) + epsilon_4[i] for i in range(num_obs)]

responses_collected = np.array([response_1,response_2,response_3,response_4])

print(responses_collected[0,:])
#Next, we will need to create the R matrices for each of the responses
R1,LSE1 = construct_R(interaction_effects,responses_collected[0,:])
R2,LSE2 = construct_R(interaction_effects,responses_collected[1,:])
R3,LSE3 = construct_R(interaction_effects,responses_collected[2,:])
R4,LSE4 = construct_R(interaction_effects,responses_collected[3,:])

R_matrices = [R1,R2,R3,R4]

#decomp_orthog takes in as input a JxN matrix of responses, a list of the R matrices for each response, a list of lists of lists for the strong heredity
#constraints, a number of clusters K, the amount of time (in seconds) for optimization. focus and outputflag are Gurobi specific parameters.
decomp = decomp_orthog(responses_collected,R_matrices,int_effect_strong_heredity,2,t=60,focus = 0, outputflag = 1)

[ 7.39085936e-01  5.99847656e+00  1.82444365e+00  2.28363170e+00
  3.67784333e+00 -4.33610349e-01  1.07710817e+00  2.71042922e+00
  4.61604936e-01 -4.08892835e-01  3.81754829e-01 -2.47680478e+00
  5.75209524e-01  6.50741774e-01  1.50950019e-01  3.35407311e+00
  3.24067263e-01  1.07528183e+00  7.31221244e+00  2.11724511e+00
  2.52663932e+00  9.62062821e-01  1.05306954e+00 -2.75288031e-01
 -4.07965207e-01  1.98050640e+00  5.47661281e+00  1.30719132e-01
  1.82807584e+00  3.06727575e+00  6.06769055e+00  6.94787854e-01
 -5.96229019e-01  7.98284147e-01 -1.37204164e+00  1.11783021e+00
  4.36396166e-01 -4.90353901e-01 -2.82703939e-02  1.66862287e+00
  4.67811829e+00  1.04969080e+00 -1.72186744e+00  1.23351817e+00
  2.47608274e+00  1.98557801e-01 -8.96146392e-01  1.32007102e+00
  2.15691452e+00 -2.29506124e+00  1.19306080e+00  6.34143134e-02
 -1.98494200e-01 -4.29223472e-01  1.82301445e+00  2.59999721e+00
 -3.53891086e-01  2.66008475e+00  1.71508341e+00  3.52775529e+00
  2.33447519e+00  6.33882

In [18]:
#Lastly, we can look at the scaling coefficients
print('scaling coefficients: ' + str(decomp[0]))

#As well as the cluster assignment
print('cluster assignment: ' + str(decomp[1]))

#And, we can scale the original LSEs by the scaling coefficients
print('scaled LSEs: ' + str(decomp[0]*[LSE1,LSE2,LSE3,LSE4]))

scaling coefficients: [[0.99705287 0.99705287 1.05422915 0.         0.         0.
  0.96675097 0.99705287 1.05422915 0.         0.         0.
  0.99705287 0.98750835 0.         0.         0.         0.9639521
  0.         0.         0.         0.         0.         0.
  0.         0.         0.        ]
 [1.04480233 1.00625204 1.04042041 0.         0.         0.
  1.04480233 1.00625204 1.04042041 0.         0.         0.
  0.96650795 0.9837409  0.         0.         0.         0.99499117
  0.         0.         0.         0.         0.         0.
  0.         0.         0.        ]
 [0.         0.         0.         1.0035522  1.04342081 1.01572198
  0.         0.         0.         1.0035522  1.04342081 0.88698514
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  1.0035522  1.0035522  1.01572198]
 [0.         0.         0.         1.08076927 0.99395142 1.0413422
  0.         0.         0.         1.0260336  0.9939